# A table you fit without the houses you score

Hold out every training house whose `Id` is divisible by 5. That is $1460/5 = 292$ houses. The other 1168 are the fit set. This cut is a fixed rule about the identifier. It is not a claim that the grades were chosen in ignorance of the whole file. The earlier lessons described all 1460 prices. The score below uses only the fit set to build the prediction.

Two constants, both taken from the 1168:

- the arithmetic mean of their prices
- the geometric mean, $\exp$ of the mean of $\ln$ price

And one table: inside the fit set, the median price of each `OverallQual` grade. A held-out house is predicted at the median of its own grade.

On the 292 held-out houses the root mean squared log errors are about

$$
\begin{aligned}
\text{arithmetic mean} &\approx 0.395 \\
\text{geometric mean} &\approx 0.383 \\
\text{grade median} &\approx 0.224
\end{aligned}
$$

The grade table cuts the log error nearly in half relative to either constant. The geometric mean beats the arithmetic mean, by a small amount, which is the same direction as the three toy prices.

The fit-set medians are not the medians from the quality lesson. Grade 7 there was 200141 on all 319 houses. On the fit set the grade-7 median is 200050. Scoring the full-file median on the holdout would let those held-out prices help write the table.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# Fit constants and grade medians on 1168 houses, score 292.
import math
from fractions import Fraction
from collections import defaultdict
fit_rows = [row for row in train if int(row["Id"]) % 5 != 0]
held_rows = [row for row in train if int(row["Id"]) % 5 == 0]
print(len(fit_rows), len(held_rows))
assert (len(fit_rows), len(held_rows)) == (1168, 292)

def rmsle(predictions, actual):
    total = 0.0
    for pred, price in zip(predictions, actual):
        gap = math.log(pred) - math.log(price)
        total += gap * gap
    return math.sqrt(total / len(actual))

def median(values):
    ordered = sorted(values)
    count = len(ordered)
    if count % 2 == 1:
        return ordered[count // 2]
    return (ordered[count // 2 - 1] + ordered[count // 2]) / 2

fit_prices = [int(row["SalePrice"]) for row in fit_rows]
held_prices = [int(row["SalePrice"]) for row in held_rows]
arithmetic = sum(fit_prices) / len(fit_prices)
geometric = math.exp(sum(math.log(price) for price in fit_prices) / len(fit_prices))
groups = defaultdict(list)
for row in fit_rows:
    groups[int(row["OverallQual"])].append(int(row["SalePrice"]))
grade_median = {grade: median(values) for grade, values in groups.items()}
print(dict(sorted(grade_median.items())))
table_pred = [grade_median[int(row["OverallQual"])] for row in held_rows]
mean_error = rmsle([arithmetic] * 292, held_prices)
geo_error = rmsle([geometric] * 292, held_prices)
table_error = rmsle(table_pred, held_prices)
print(mean_error, geo_error, table_error)
assert grade_median[7] == 200050
assert abs(mean_error - 0.394704176512337) < 1e-12
assert abs(geo_error - 0.3829992574136876) < 1e-12
assert abs(table_error - 0.2237289381904309) < 1e-12
assert geo_error < mean_error < 0.5
assert table_error < geo_error


1168 292
{1: 50150.0, 2: 60000, 3: 82000, 4: 108959, 5: 134900, 6: 159000.0, 7: 200050.0, 8: 268000, 9: 345000, 10: 445365.0}
0.394704176512337 0.3829992574136876 0.2237289381904309


## Pitfall

A table of 10 medians is a model. If the median is computed on all 1460 rows and then scored on a subset, the subset helped set its own prediction. The 0.224 above uses medians from the other 1168 houses only.

## Summary

Remainder 0 of `Id` holds out 292 houses. On those houses a fit-set grade median has RMSLE about 0.224. The arithmetic mean of the fit set has about 0.395, and the geometric mean about 0.383.
